# Restaurant AI - Visualization

This notebook provides visualization and plotting capabilities for analytics data.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime

from utils.visualization import (
    AnalyticsEngine, plot_footfall_graph,
    plot_wait_time_distribution, plot_staff_activity
)

## 2. Load Data

In [ ]:
# Load analytics report
report_path = Path('../outputs/analytics_report.json')

if report_path.exists():
    with open(report_path, 'r') as f:
        report = json.load(f)
    
    print("Report loaded successfully!")
    print(f"Total entries: {report['summary']['total_entries']}")
    print(f"Total exits: {report['summary']['total_exits']}")
    print(f"Average wait time: {report['summary']['avg_wait_time']:.1f}s")
else:
    print("No report found. Run analytics.ipynb first.")
    report = None

## 3. Footfall Visualization

In [ ]:
if report and report['data']['footfall']:
    footfall_data = report['data']['footfall']
    
    # Using plotly for interactive chart
    fig = plot_footfall_graph(footfall_data)
    fig.show()
    
    # Also create matplotlib version
    df = pd.DataFrame(footfall_data)
    df['time'] = pd.to_datetime(df['timestamp'], unit='s')
    
    plt.figure(figsize=(12, 6))
    plt.plot(df['time'], df['entries'], label='Entries', marker='o')
    plt.plot(df['time'], df['exits'], label='Exits', marker='x')
    plt.plot(df['time'], df['current'], label='Current', linewidth=2)
    plt.xlabel('Time')
    plt.ylabel('Count')
    plt.title('Footfall Over Time')
    plt.legend()
    plt.grid(True)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 4. Wait Time Distribution

In [ ]:
if report and report['data']['wait_times']:
    wait_times = [w['wait_time'] for w in report['data']['wait_times']]
    
    # Plotly histogram
    fig = plot_wait_time_distribution(wait_times)
    fig.show()
    
    # Matplotlib histogram
    plt.figure(figsize=(10, 6))
    plt.hist(wait_times, bins=30, color='skyblue', edgecolor='black')
    plt.xlabel('Wait Time (seconds)')
    plt.ylabel('Frequency')
    plt.title('Wait Time Distribution')
    plt.axvline(np.mean(wait_times), color='red', linestyle='--', label=f'Mean: {np.mean(wait_times):.1f}s')
    plt.axvline(np.median(wait_times), color='green', linestyle='--', label=f'Median: {np.median(wait_times):.1f}s')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"\nWait Time Statistics:")
    print(f"  Mean: {np.mean(wait_times):.1f}s ({np.mean(wait_times)/60:.1f} min)")
    print(f"  Median: {np.median(wait_times):.1f}s ({np.median(wait_times)/60:.1f} min)")
    print(f"  Min: {np.min(wait_times):.1f}s")
    print(f"  Max: {np.max(wait_times):.1f}s")

## 5. Peak Hours Heatmap

In [ ]:
if report and report['summary']['peak_hours']:
    peak_data = report['summary']['peak_hours']
    hourly_dist = peak_data.get('hourly_distribution', {})
    
    # Create heatmap data
    hours = list(range(24))
    counts = [hourly_dist.get(h, 0) for h in hours]
    
    # Reshape for heatmap (6 hours per row)
    heatmap_data = np.array(counts).reshape(4, 6)
    
    plt.figure(figsize=(10, 6))
    plt.imshow(heatmap_data, cmap='YlOrRd', aspect='auto')
    plt.colorbar(label='Entries')
    
    # Add labels
    hour_labels = [f'{h}:00' for h in range(0, 24, 4)]
    plt.xticks(range(6), hour_labels)
    plt.yticks(range(4), ['00-05', '06-11', '12-17', '18-23'])
    
    plt.xlabel('Time Period')
    plt.ylabel('Period')
    plt.title('Peak Hours Heatmap')
    
    # Add value annotations
    for i in range(4):
        for j in range(6):
            plt.text(j, i, heatmap_data[i, j], ha='center', va='center', color='white')
    
    plt.tight_layout()
    plt.show()

## 6. Zone Duration Analysis

In [ ]:
if report and report['data']['zone_durations']:
    zone_data = report['data']['zone_durations']
    
    zones = list(zone_data.keys())
    durations = [np.mean(zone_data[z]) if zone_data[z] else 0 for z in zones]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(zones, durations, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    plt.xlabel('Zone')
    plt.ylabel('Average Duration (seconds)')
    plt.title('Average Time Spent in Each Zone')
    
    # Add value labels
    for bar, duration in zip(bars, durations):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{duration:.1f}s', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

## 7. Dashboard Summary

In [ ]:
# Create summary dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Restaurant Analytics Dashboard', fontsize=16, fontweight='bold')

# 1. Footfall summary
ax1 = axes[0, 0]
if report:
    summary = report['summary']
    ax1.bar(['Entries', 'Exits'], [summary['total_entries'], summary['total_exits']], 
            color=['#2ecc71', '#e74c3c'])
    ax1.set_title('Total Footfall')
    ax1.set_ylabel('Count')

# 2. Wait time distribution
ax2 = axes[0, 1]
if report and report['data']['wait_times']:
    wait_times = [w['wait_time'] for w in report['data']['wait_times']]
    ax2.hist(wait_times, bins=20, color='#3498db', edgecolor='black')
    ax2.set_title('Wait Time Distribution')
    ax2.set_xlabel('Wait Time (s)')
    ax2.set_ylabel('Frequency')

# 3. Zone analysis
ax3 = axes[1, 0]
if report and report['data']['zone_durations']:
    zone_data = report['data']['zone_durations']
    zones = list(zone_data.keys())
    durations = [np.mean(zone_data[z]) if zone_data[z] else 0 for z in zones]
    ax3.bar(zones, durations, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    ax3.set_title('Average Zone Duration')
    ax3.set_ylabel('Seconds')

# 4. Peak hours
ax4 = axes[1, 1]
if report and report['summary']['peak_hours']:
    peak_data = report['summary']['peak_hours']
    hourly_dist = peak_data.get('hourly_distribution', {})
    hours = list(range(24))
    counts = [hourly_dist.get(h, 0) for h in hours]
    ax4.fill_between(hours, counts, alpha=0.3, color='#9b59b6')
    ax4.plot(hours, counts, color='#9b59b6', linewidth=2)
    ax4.set_title('Hourly Distribution')
    ax4.set_xlabel('Hour')
    ax4.set_ylabel('Entries')
    ax4.set_xticks(range(0, 24, 4))

plt.tight_layout()
plt.show()

## 8. Export Visualizations

In [ ]:
# Save visualizations
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

if report and report['data']['footfall']:
    plot_footfall_graph(report['data']['footfall'], str(output_dir / 'footfall_graph.html'))
    print(f"Saved footfall graph to {output_dir / 'footfall_graph.html'}")

if report and report['data']['wait_times']:
    wait_times = [w['wait_time'] for w in report['data']['wait_times']]
    plot_wait_time_distribution(wait_times, str(output_dir / 'wait_time_dist.html'))
    print(f"Saved wait time distribution to {output_dir / 'wait_time_dist.html'}")

## Summary

This notebook covers:
- Loading analytics reports
- Interactive footfall graphs using Plotly
- Wait time distribution visualization
- Peak hours heatmap
- Zone duration analysis
- Complete dashboard summary
- Export to HTML for sharing